COST BY SERVICE

In [0]:
df_gold_cost_by_service = spark.sql("""
    SELECT
        RECORD_TYPE,
        CHARGE_CODE,
        CATEGORY,
        
        SUM(QUANTITY)       as TOTAL_QUANTITY,
        SUM(VENDOR_COST)    as TOTAL_VENDOR_COST,
        SUM(TOTAL_CHARGES)  as TOTAL_CHARGES
    FROM (
        SELECT RECORD_TYPE, CHARGE_CODE, CATEGORY, QUANTITY, VENDOR_COST, TOTAL_CHARGES
        FROM finops.silver.focus

        UNION ALL

        SELECT RECORD_TYPE, CHARGE_CODE, CATEGORY, QUANTITY, VENDOR_COST, TOTAL_CHARGES
        FROM finops.silver.non_focus
    )
    GROUP BY RECORD_TYPE, CHARGE_CODE, CATEGORY
    ORDER BY RECORD_TYPE, CHARGE_CODE, CATEGORY
""")

print(f"✅ Gold aggregation done — {df_gold_cost_by_service.count()} rows")
df_gold_cost_by_service.show(10)

In [0]:
df_gold_cost_by_service.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://gold@adbstoragev10.dfs.core.windows.net/cost_by_service")

MONTHLY TREND

In [0]:
df_gold_monthly_trend = spark.sql("""
    SELECT
        RECORD_TYPE,
        DATE_TRUNC('month', TO_DATE(USAGE_DATE, 'yyyyMMdd'))  as BILLING_MONTH,
        CATEGORY,
        CHARGE_CODE,
        SUM(VENDOR_COST)                                         as TOTAL_VENDOR_COST,
        SUM(TOTAL_CHARGES)                                       as TOTAL_CHARGES,
        SUM(QUANTITY)                                            as TOTAL_QUANTITY
    FROM (
        SELECT RECORD_TYPE,USAGE_DATE, CATEGORY, CHARGE_CODE, VENDOR_COST, TOTAL_CHARGES, QUANTITY
        FROM finops.silver.focus
        UNION ALL
        SELECT RECORD_TYPE,USAGE_DATE, CATEGORY, CHARGE_CODE, VENDOR_COST, TOTAL_CHARGES, QUANTITY
        FROM finops.silver.non_focus
    )
    GROUP BY DATE_TRUNC('month', TO_DATE(USAGE_DATE, 'yyyyMMdd')), CATEGORY, CHARGE_CODE,RECORD_TYPE
    ORDER BY RECORD_TYPE,BILLING_MONTH, TOTAL_CHARGES DESC
""")

In [0]:
df_gold_monthly_trend.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://gold@adbstoragev10.dfs.core.windows.net/monthly_trend")

df_gold_monthly_trend.show()

COST BY ACCOUNT

In [0]:
df_gold_cost_by_account = spark.sql("""
    SELECT
        RECORD_TYPE,
        PARENT_ACCOUNT_NUMBER,
        ACCOUNT_NUMBER,
        ACCOUNT_NAME,
        DATE_TRUNC('month', TO_DATE(USAGE_DATE, 'yyyyMMdd'))  as BILLING_MONTH,
        SUM(VENDOR_COST)                 as TOTAL_VENDOR_COST,
        SUM(TOTAL_CHARGES)               as TOTAL_CHARGES,
        COUNT(DISTINCT CHARGE_CODE)      as CLOUD_SERVICES
    FROM (
        SELECT RECORD_TYPE,PARENT_ACCOUNT_NUMBER, ACCOUNT_NUMBER, ACCOUNT_NAME, USAGE_DATE, VENDOR_COST, TOTAL_CHARGES, CHARGE_CODE
        FROM finops.silver.focus
        UNION ALL
        SELECT RECORD_TYPE,PARENT_ACCOUNT_NUMBER, ACCOUNT_NUMBER, ACCOUNT_NAME, USAGE_DATE, VENDOR_COST, TOTAL_CHARGES, CHARGE_CODE
        FROM finops.silver.non_focus
    )
    GROUP BY PARENT_ACCOUNT_NUMBER, ACCOUNT_NUMBER, ACCOUNT_NAME, DATE_TRUNC('month', TO_DATE(USAGE_DATE, 'yyyyMMdd')),RECORD_TYPE
    ORDER BY RECORD_TYPE, BILLING_MONTH, TOTAL_CHARGES DESC
""")

In [0]:
df_gold_cost_by_account.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://gold@adbstoragev10.dfs.core.windows.net/cost_by_account")

df_gold_cost_by_account.show()

COST BY REGION

In [0]:
df_gold_cost_by_region = spark.sql("""
    SELECT
        RECORD_TYPE,
        REGION,
        REGION_NAME,
        CATEGORY,
        SUM(VENDOR_COST)                                        as TOTAL_VENDOR_COST,
        SUM(TOTAL_CHARGES)                                      as TOTAL_CHARGES,
        ROUND(SUM(TOTAL_CHARGES) * 100.0 /
              SUM(SUM(TOTAL_CHARGES)) OVER (), 2)               as PCT_OF_TOTAL
    FROM (
        SELECT RECORD_TYPE,REGION, REGION_NAME, CATEGORY, VENDOR_COST, TOTAL_CHARGES
        FROM finops.silver.focus
        UNION ALL
        SELECT RECORD_TYPE,REGION, REGION_NAME, CATEGORY, VENDOR_COST, TOTAL_CHARGES
        FROM finops.silver.non_focus
    )
    GROUP BY RECORD_TYPE,REGION, REGION_NAME, CATEGORY
    ORDER BY RECORD_TYPE,TOTAL_CHARGES DESC
""")

In [0]:
df_gold_cost_by_region.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://gold@adbstoragev10.dfs.core.windows.net/cost_by_region")

df_gold_cost_by_region.show()

TAG ALLOCATION

In [0]:
df_gold_tag_allocation = spark.sql("""
    SELECT
        RECORD_TYPE,
        RESOURCEGROUP,
        BUSINESS_UNIT,
        COALESCE(COST_CENTER, ORIGIN_COST_CENTER, 'UNALLOCATED') as COST_CENTER,
        CASE WHEN TAGS IS NULL OR TAGS = '{}' OR TAGS = ''
             THEN 'UNTAGGED' ELSE 'TAGGED' END                  as TAG_STATUS,
        SUM(TOTAL_CHARGES)                                       as TOTAL_CHARGES,
        COUNT(*)                                                 as RECORD_COUNT
    FROM (
        SELECT RECORD_TYPE, RESOURCEGROUP, BUSINESS_UNIT, COST_CENTER,
               ORIGIN_COST_CENTER, TAGS, TOTAL_CHARGES
        FROM finops.silver.focus
        UNION ALL
        SELECT RECORD_TYPE, RESOURCEGROUP, BUSINESS_UNIT, COST_CENTER,
               ORIGIN_COST_CENTER, TAGS, TOTAL_CHARGES
        FROM finops.silver.non_focus
    )
    GROUP BY RECORD_TYPE,RESOURCEGROUP, BUSINESS_UNIT,
             COALESCE(COST_CENTER, ORIGIN_COST_CENTER, 'UNALLOCATED'),
             CASE WHEN TAGS IS NULL OR TAGS = '{}' OR TAGS = '' THEN 'UNTAGGED' ELSE 'TAGGED' END
    ORDER BY RECORD_TYPE,TOTAL_CHARGES DESC
""")

In [0]:
df_gold_tag_allocation.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://gold@adbstoragev10.dfs.core.windows.net/tag_allocation")

df_gold_tag_allocation.show()